# 🏀 Over/Under — Previsão de Total de Pontos

**Objetivo:** Prever se o total de pontos de um jogo NBA ficará acima (Over) ou abaixo (Under) de um determinado valor.

## Pipeline
1. Buscar TODOS os jogos da temporada via API (sem afetar o banco)
2. Construir dataset de confrontos (features dos dois times)
3. Feature engineering
4. Treinar e comparar modelos
5. Avaliar resultados

## Modelos
- **Logistic Regression** — baseline interpretável
- **Random Forest** — robusto, pouco overfitting
- **XGBoost** — tipicamente o melhor para dados tabulares
- **Ridge Regression** — abordagem de regressão (prever total → classificar)

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import cross_val_score, StratifiedKFold, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, mean_absolute_error
)
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠️ XGBoost não instalado. Instale com: pip install xgboost")

sns.set_theme(style="whitegrid", font_scale=1.1)
pd.set_option("display.max_columns", 50)

DB_PATH = Path("data/nba.db")
print(f"✅ Imports OK | DB exists: {DB_PATH.exists()}")

✅ Imports OK | DB exists: True


## 1. Buscar TODOS os jogos da temporada via API

Busca o game log completo de cada time (todos os jogos, não só os últimos 15).  
Usa a mesma API do `nba_data.py` mas **não toca no banco** — tudo fica em memória.

In [ ]:
import time
from nba_api.stats.static import teams as nba_teams_static
from nba_api.stats.endpoints import (
    TeamGameLog, TeamEstimatedMetrics, LeagueDashTeamStats,
    TeamDashboardByGeneralSplits,
)

SEASON = "2025-26"
SLEEP = 0.8
TIMEOUT = 60

# 1) Lista de times
all_nba_teams = nba_teams_static.get_teams()
team_id_to_abbr = {t["id"]: t["abbreviation"] for t in all_nba_teams}
print(f"Times: {len(all_nba_teams)}")

# 2) Buscar game log COMPLETO de cada time
all_rows = []
for i, tm in enumerate(all_nba_teams):
    tid, abbr = tm["id"], tm["abbreviation"]
    print(f"  [{i+1:2d}/30] {abbr}...", end=" ", flush=True)
    time.sleep(SLEEP)
    log = TeamGameLog(team_id=tid, season=SEASON,
                      season_type_all_star="Regular Season", timeout=TIMEOUT)
    df = log.get_data_frames()[0]
    for _, row in df.iterrows():
        all_rows.append({
            "game_id": str(row["Game_ID"]),
            "team_abbr": abbr,
            "team_id": tid,
            "date": str(row["GAME_DATE"]),
            "matchup": str(row["MATCHUP"]),
            "wl": str(row["WL"]),
            "pts": int(row["PTS"]),
            "oreb": int(row.get("OREB", 0)),
            "dreb": int(row.get("DREB", 0)),
            "reb": int(row["REB"]),
            "ast": int(row["AST"]),
            "stl": int(row.get("STL", 0)),
            "blk": int(row.get("BLK", 0)),
            "tov": int(row.get("TOV", 0)),
            "pf": int(row.get("PF", 0)),
            "fgm": int(row.get("FGM", 0)),
            "fga": int(row.get("FGA", 0)),
            "fg_pct": round(float(row["FG_PCT"]) * 100, 1),
            "fg3m": int(row.get("FG3M", 0)),
            "fg3a": int(row.get("FG3A", 0)),
            "fg3_pct": round(float(row.get("FG3_PCT", 0)) * 100, 1),
            "ftm": int(row.get("FTM", 0)),
            "fta": int(row.get("FTA", 0)),
            "ft_pct": round(float(row.get("FT_PCT", 0)) * 100, 1),
        })
    print(f"{len(df)} jogos")

df_games = pd.DataFrame(all_rows)
print(f"\n✅ Games: {len(df_games)} linhas | {df_games['game_id'].nunique()} jogos únicos")
print(f"Período: {df_games['date'].min()} → {df_games['date'].max()}")

# 3) Buscar métricas avançadas da API (pace, off/def rating, etc.)
print("\n📊 Buscando métricas avançadas da API...")

print("  Estimated metrics (pace, ratings)...", end=" ", flush=True)
time.sleep(SLEEP)
est = TeamEstimatedMetrics(season=SEASON, season_type="Regular Season", timeout=TIMEOUT)
df_est = est.get_data_frames()[0]
advanced = {}
for _, row in df_est.iterrows():
    tid = int(row["TEAM_ID"])
    advanced[team_id_to_abbr.get(tid, "")] = {
        "off_rating": round(float(row["E_OFF_RATING"]), 1),
        "def_rating": round(float(row["E_DEF_RATING"]), 1),
        "net_rating": round(float(row["E_NET_RATING"]), 1),
        "pace": round(float(row["E_PACE"]), 1),
        "ast_ratio": round(float(row["E_AST_RATIO"]), 1),
        "oreb_pct": round(float(row["E_OREB_PCT"]) * 100, 1),
        "dreb_pct": round(float(row["E_DREB_PCT"]) * 100, 1),
        "reb_pct": round(float(row["E_REB_PCT"]) * 100, 1),
        "tov_pct": round(float(row["E_TM_TOV_PCT"]) * 100, 1),
    }
print("ok")

print("  Opponent stats...", end=" ", flush=True)
time.sleep(SLEEP)
opp = LeagueDashTeamStats(season=SEASON, per_mode_detailed="PerGame",
                          measure_type_detailed_defense="Opponent", timeout=TIMEOUT)
df_opp = opp.get_data_frames()[0]
for _, row in df_opp.iterrows():
    abbr = team_id_to_abbr.get(int(row["TEAM_ID"]), "")
    if abbr in advanced:
        advanced[abbr].update({
            "opp_pts": round(float(row["OPP_PTS"]), 1),
            "opp_fg_pct": round(float(row["OPP_FG_PCT"]) * 100, 1),
            "opp_fg3_pct": round(float(row["OPP_FG3_PCT"]) * 100, 1),
        })
print("ok")

print("  Misc stats (paint, fast break, 2nd chance)...", end=" ", flush=True)
time.sleep(SLEEP)
misc = LeagueDashTeamStats(season=SEASON, per_mode_detailed="PerGame",
                           measure_type_detailed_defense="Misc", timeout=TIMEOUT)
df_misc = misc.get_data_frames()[0]
for _, row in df_misc.iterrows():
    abbr = team_id_to_abbr.get(int(row["TEAM_ID"]), "")
    if abbr in advanced:
        advanced[abbr].update({
            "pts_paint": round(float(row["PTS_PAINT"]), 1),
            "pts_fb": round(float(row["PTS_FB"]), 1),
            "pts_2nd_chance": round(float(row["PTS_2ND_CHANCE"]), 1),
        })
print("ok")

# Adicionar médias de temporada (pts, ast, fga, etc.) calculadas do game log
for abbr in team_id_to_abbr.values():
    team_games = df_games[df_games["team_abbr"] == abbr]
    if len(team_games) == 0:
        continue
    if abbr not in advanced:
        advanced[abbr] = {}
    advanced[abbr].update({
        "pts": round(team_games["pts"].mean(), 1),
        "ast": round(team_games["ast"].mean(), 1),
        "reb": round(team_games["reb"].mean(), 1),
        "oreb": round(team_games["oreb"].mean(), 1),
        "tov": round(team_games["tov"].mean(), 1),
        "fga": round(team_games["fga"].mean(), 1),
        "fg3a": round(team_games["fg3a"].mean(), 1),
        "fta": round(team_games["fta"].mean(), 1),
        "fg_pct": round(team_games["fg_pct"].mean(), 1),
        "fg3_pct": round(team_games["fg3_pct"].mean(), 1),
        "ft_pct": round(team_games["ft_pct"].mean(), 1),
    })

df_teams = pd.DataFrame.from_dict(advanced, orient="index")
df_teams.index.name = "abbreviation"
df_teams = df_teams.reset_index()

print(f"\n✅ Teams: {len(df_teams)} times com {len(df_teams.columns)} features")
print(f"Colunas: {list(df_teams.columns)}")

Times: 30
  [ 1/30] ATL... 76 jogos
  [ 2/30] BOS... 75 jogos
  [ 3/30] CLE... 75 jogos
  [ 4/30] NOP... 76 jogos
  [ 5/30] CHI... 75 jogos
  [ 6/30] DAL... 75 jogos
  [ 7/30] DEN... 76 jogos
  [ 8/30] GSW... 75 jogos
  [ 9/30] HOU... 74 jogos
  [10/30] LAC... 75 jogos
  [11/30] LAL... 75 jogos
  [12/30] MIA... 76 jogos
  [13/30] MIL... 74 jogos
  [14/30] MIN... 75 jogos
  [15/30] BKN... 75 jogos
  [16/30] NYK... 75 jogos
  [17/30] ORL... 74 jogos
  [18/30] IND... 75 jogos
  [19/30] PHI... 75 jogos
  [20/30] PHX... 75 jogos
  [21/30] POR... 76 jogos
  [22/30] SAC... 76 jogos
  [23/30] SAS... 75 jogos
  [24/30] OKC... 76 jogos
  [25/30] TOR... 74 jogos
  [26/30] UTA... 76 jogos
  [27/30] MEM... 75 jogos
  [28/30] WAS... 75 jogos
  [29/30] DET... 75 jogos
  [30/30] CHA... 75 jogos

✅ Total: 2254 linhas | 1127 jogos únicos
Período: DEC 01, 2025 → OCT 31, 2025
Teams (banco): 30 times


In [3]:
# Visualizar estrutura dos dados
print(f"=== GAMES ({len(df_games)} linhas, {df_games['game_id'].nunique()} jogos) ===")
display(df_games.head(10))
print(f"\nColunas: {list(df_games.columns)}")
print(f"\nJogos por time:")
print(df_games.groupby('team_abbr').size().sort_values(ascending=False).to_string())
print(f"\nStats descritivas dos pontos:")
display(df_games['pts'].describe())

=== GAMES (2254 linhas, 1127 jogos) ===


,game_id,team_abbr,team_id,date,matchup,wl,pts,oreb,dreb,reb,ast,stl,blk,tov,pf,fgm,fga,fg_pct,fg3m,fg3a,fg3_pct,ftm,fta,ft_pct
0,0022501092,ATL,1610612737,"MAR 30, 2026",ATL vs. BOS,W,112,9,36,45,28,7,6,11,21,43,92,46.7,15,38,39.5,11,15,73.3
1,0022501078,ATL,1610612737,"MAR 28, 2026",ATL vs. SAC,W,123,13,27,40,34,10,3,13,12,45,89,50.6,16,41,39.0,17,22,77.3
2,0022501067,ATL,1610612737,"MAR 27, 2026",ATL @ BOS,L,102,10,25,35,23,6,4,4,21,34,87,39.1,15,42,35.7,19,24,79.2
3,0022501051,ATL,1610612737,"MAR 25, 2026",ATL @ DET,W,130,18,32,50,31,8,1,20,19,50,105,47.6,18,52,34.6,12,17,70.6
4,0022501037,ATL,1610612737,"MAR 23, 2026",ATL vs. MEM,W,146,12,32,44,37,12,6,11,19,49,92,53.3,25,54,46.3,23,25,92.0
5,0022501026,ATL,1610612737,"MAR 21, 2026",ATL vs. GSW,W,126,9,32,41,28,10,4,18,17,48,89,53.9,13,40,32.5,17,20,85.0
6,0022501018,ATL,1610612737,"MAR 20, 2026",ATL @ HOU,L,95,9,28,37,22,13,9,16,21,36,85,42.4,9,35,25.7,14,17,82.4
7,0022501006,ATL,1610612737,"MAR 18, 2026",ATL @ DAL,W,135,12,30,42,36,10,3,11,21,55,102,53.9,14,35,40.0,11,14,78.6
8,0022500983,ATL,1610612737,"MAR 16, 2026",ATL vs. ORL,W,124,15,39,54,33,5,3,13,20,43,93,46.2,16,40,40.0,22,29,75.9
9,0022500970,ATL,1610612737,"MAR 14, 2026",ATL vs. MIL,W,122,15,32,47,29,10,1,13,19,47,95,49.5,14,38,36.8,14,18,77.8



Colunas: ['game_id', 'team_abbr', 'team_id', 'date', 'matchup', 'wl', 'pts', 'oreb', 'dreb', 'reb', 'ast', 'stl', 'blk', 'tov', 'pf', 'fgm', 'fga', 'fg_pct', 'fg3m', 'fg3a', 'fg3_pct', 'ftm', 'fta', 'ft_pct']

Jogos por time:
team_abbr
ATL    76
DEN    76
SAC    76
OKC    76
NOP    76
MIA    76
UTA    76
POR    76
CLE    75
DAL    75
BOS    75
CHA    75
CHI    75
BKN    75
PHI    75
LAL    75
LAC    75
IND    75
GSW    75
DET    75
NYK    75
MEM    75
WAS    75
SAS    75
MIN    75
PHX    75
HOU    74
MIL    74
ORL    74
TOR    74

Stats descritivas dos pontos:


count    2254.000000
mean      115.438776
std        12.753233
min        66.000000
25%       107.000000
50%       115.000000
75%       124.000000
max       157.000000
Name: pts, dtype: float64

## 2. Construir dataset de confrontos

Cada jogo tem 2 linhas no banco (uma por time). Vamos montar **1 linha por jogo** com features dos dois times.

In [ ]:
# Parear as duas linhas de cada jogo
# Identificar home/away pelo matchup ("vs." = home, "@" = away)
df_games["is_home"] = df_games["matchup"].str.contains("vs.").astype(int)

# Separar home e away
home = df_games[df_games["is_home"] == 1].copy()
away = df_games[df_games["is_home"] == 0].copy()

print(f"Home games: {len(home)} | Away games: {len(away)}")

# Merge por game_id para criar 1 linha por jogo
matchups = home.merge(
    away,
    on="game_id",
    suffixes=("_home", "_away"),
    how="inner"
)

# Total de pontos
matchups["total_pts"] = matchups["pts_home"] + matchups["pts_away"]

print(f"\nConfrontos montados: {len(matchups)}")
print(f"Total de pontos — média: {matchups['total_pts'].mean():.1f}, std: {matchups['total_pts'].std():.1f}")
display(matchups[["date_home", "team_abbr_home", "team_abbr_away", "pts_home", "pts_away", "total_pts"]].head(10))

## 3. Feature Engineering

Vamos enriquecer cada confronto com as **médias da temporada** de cada time (da tabela `teams`).

Features principais:
- **Pace** dos dois times → número de posses
- **Off/Def Rating** → eficiência ofensiva/defensiva
- **FG%, 3P%, FT%** → eficiência de arremesso
- **Pontos no garrafão, transição, 2ª chance** → perfil ofensivo
- **Turnovers, rebotes ofensivos** → posses extras
- **Pontos da temporada (equipe + adversário)** → baseline de pontuação

In [ ]:
# Features da temporada para cada time
team_features = [
    "pts", "opp_pts", "pace", "off_rating", "def_rating", "net_rating",
    "fg_pct", "fg3_pct", "ft_pct", "fga", "fg3a", "fta",
    "tov", "tov_pct", "oreb", "oreb_pct",
    "pts_paint", "pts_fb", "pts_2nd_chance", "pts_mid_range",
    "ast", "ast_ratio",
    "opp_fg_pct", "opp_fg3_pct",
]

# Filtrar colunas que existem
team_features = [f for f in team_features if f in df_teams.columns]
print(f"Features disponíveis: {len(team_features)}")
print(team_features)

# Preparar lookup de features por time
teams_lookup = df_teams.set_index("abbreviation")[team_features].to_dict(orient="index")

In [ ]:
# Adicionar features da temporada ao dataset de confrontos
for side in ["home", "away"]:
    abbr_col = f"team_abbr_{side}"
    for feat in team_features:
        matchups[f"{feat}_{side}"] = matchups[abbr_col].map(
            lambda a, f=feat: teams_lookup.get(a, {}).get(f)
        )

# Features derivadas (combinação dos dois times)
matchups["combined_pace"] = (matchups["pace_home"] + matchups["pace_away"]) / 2
matchups["combined_pts_avg"] = matchups["pts_home"] + matchups["pts_away"]  # do jogo
matchups["season_pts_avg"] = (
    matchups["pts_home"].astype(float) + matchups["pts_away"].astype(float)
)  # será sobrescrito abaixo
matchups["implied_total"] = (
    (matchups["off_rating_home"] + matchups["off_rating_away"]) * matchups["combined_pace"] / 100
)
matchups["off_diff"] = matchups["off_rating_home"] - matchups["off_rating_away"]
matchups["def_diff"] = matchups["def_rating_home"] - matchups["def_rating_away"]
matchups["pace_diff"] = matchups["pace_home"] - matchups["pace_away"]
matchups["fg_pct_diff"] = matchups["fg_pct_home"] - matchups["fg_pct_away"]
matchups["tov_sum"] = matchups["tov_home"] + matchups["tov_away"]
matchups["oreb_sum"] = matchups["oreb_home"] + matchups["oreb_away"]
matchups["pts_paint_sum"] = matchups["pts_paint_home"] + matchups["pts_paint_away"]
matchups["pts_fb_sum"] = matchups["pts_fb_home"] + matchups["pts_fb_away"]

print(f"Dataset final: {matchups.shape}")
print(f"\nFeatures criadas: {matchups.shape[1]} colunas")

## 4. EDA — Análise Exploratória

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribuição do total de pontos
axes[0].hist(matchups["total_pts"], bins=25, color="#1565C0", edgecolor="white", alpha=0.8)
axes[0].axvline(matchups["total_pts"].mean(), color="red", linestyle="--", label=f"μ = {matchups['total_pts'].mean():.1f}")
axes[0].axvline(matchups["total_pts"].median(), color="orange", linestyle="--", label=f"mediana = {matchups['total_pts'].median():.1f}")
axes[0].set_title("Distribuição do Total de Pontos")
axes[0].set_xlabel("Total pts")
axes[0].legend(fontsize=9)

# Total vs Pace combinado
axes[1].scatter(matchups["combined_pace"], matchups["total_pts"], alpha=0.5, color="#1565C0", s=30)
axes[1].set_title("Total pts vs Pace combinado")
axes[1].set_xlabel("Pace combinado")
axes[1].set_ylabel("Total pts")

# Total vs Implied total 
axes[2].scatter(matchups["implied_total"], matchups["total_pts"], alpha=0.5, color="#2e7d32", s=30)
# Linha de referência y=x
lims = [min(matchups["implied_total"].min(), matchups["total_pts"].min()) - 5,
        max(matchups["implied_total"].max(), matchups["total_pts"].max()) + 5]
axes[2].plot(lims, lims, "--", color="red", alpha=0.5, label="y=x")
axes[2].set_title("Total real vs Total implícito")
axes[2].set_xlabel("Implied total (OffRtg × Pace / 100)")
axes[2].set_ylabel("Total pts real")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Correlação do implied_total com total real
corr = matchups[["total_pts", "implied_total"]].corr().iloc[0, 1]
print(f"\nCorrelação implied_total vs total_pts: {corr:.3f}")

In [ ]:
# Correlação das features com o total de pontos
feature_cols = [
    "combined_pace", "implied_total", "off_diff", "def_diff", "pace_diff",
    "tov_sum", "oreb_sum", "pts_paint_sum", "pts_fb_sum",
] + [f"{f}_{s}" for f in team_features for s in ["home", "away"]]

# Filtrar só numéricas que existem
feature_cols = [c for c in feature_cols if c in matchups.columns]
corr_with_total = matchups[feature_cols + ["total_pts"]].corr()["total_pts"].drop("total_pts").sort_values(ascending=False)

print("Top 15 correlações positivas com total_pts:")
print(corr_with_total.head(15).to_string())
print("\nTop 10 correlações negativas:")
print(corr_with_total.tail(10).to_string())

In [ ]:
# Heatmap das top features
top_feats = list(corr_with_total.abs().sort_values(ascending=False).head(12).index)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    matchups[top_feats + ["total_pts"]].corr(),
    annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    square=True, ax=ax
)
ax.set_title("Correlação — Top features vs Total de Pontos")
plt.tight_layout()
plt.show()

## 5. Definir target e preparar dados

**Linha (threshold):** Usaremos a **média do total de pontos** como linha de referência.  
Em produção, isso seria substituído pela linha real de apostas (Vegas line).

- **Over** = total > linha → 1
- **Under** = total ≤ linha → 0

In [ ]:
# Definir a linha como a mediana do total de pontos
LINE = matchups["total_pts"].median()
print(f"Linha (mediana): {LINE:.1f} pontos")

matchups["over"] = (matchups["total_pts"] > LINE).astype(int)
print(f"\nDistribuição do target:")
print(matchups["over"].value_counts())
print(f"  Over:  {matchups['over'].mean()*100:.1f}%")
print(f"  Under: {(1-matchups['over'].mean())*100:.1f}%")

In [ ]:
# Selecionar features para o modelo
# Usar APENAS features que estariam disponíveis ANTES do jogo
# (médias da temporada dos dois times — NÃO stats do jogo em si)

model_features = [
    # Pace e ratings
    "combined_pace", "implied_total",
    "off_rating_home", "off_rating_away",
    "def_rating_home", "def_rating_away",
    "net_rating_home", "net_rating_away",
    "pace_home", "pace_away",
    # Pontuação média
    "pts_home", "pts_away",
    "opp_pts_home", "opp_pts_away",
    # Eficiência
    "fg_pct_home", "fg_pct_away",
    "fg3_pct_home", "fg3_pct_away",
    "ft_pct_home", "ft_pct_away",
    # Volume
    "fga_home", "fga_away",
    "fg3a_home", "fg3a_away",
    "fta_home", "fta_away",
    # Posses extras
    "tov_home", "tov_away",
    "oreb_home", "oreb_away",
    # Perfil ofensivo
    "pts_paint_home", "pts_paint_away",
    "pts_fb_home", "pts_fb_away",
    "pts_2nd_chance_home", "pts_2nd_chance_away",
    # Defesa do oponente
    "opp_fg_pct_home", "opp_fg_pct_away",
    "opp_fg3_pct_home", "opp_fg3_pct_away",
    # Combinadas
    "off_diff", "def_diff", "pace_diff",
    "tov_sum", "oreb_sum",
]

# Filtrar features que existem
model_features = [f for f in model_features if f in matchups.columns]
print(f"Features para o modelo: {len(model_features)}")

# Remover NaN
df_ml = matchups[model_features + ["over", "total_pts", "date_home"]].dropna()
print(f"Amostras válidas: {len(df_ml)} de {len(matchups)}")

X = df_ml[model_features]
y = df_ml["over"]

print(f"\nShape: X={X.shape}, y={y.shape}")
print(f"Balance: Over={y.mean()*100:.1f}% | Under={(1-y.mean())*100:.1f}%")

## 6. Treinar e comparar modelos

Agora com **todos os jogos da temporada** (~1000+ confrontos) e métricas avançadas da API (pace, off/def rating, etc.).

In [ ]:
# Definir modelos
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=42, n_jobs=-1
    ),
}

if HAS_XGB:
    models["XGBoost"] = XGBClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        random_state=42, eval_metric="logloss", verbosity=0
    )

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    results[name] = {
        "mean": scores.mean(),
        "std": scores.std(),
        "scores": scores
    }
    print(f"{name:25s}  Acc: {scores.mean():.3f} ± {scores.std():.3f}  | folds: {[f'{s:.3f}' for s in scores]}")

print(f"\n🎯 Baseline (sempre Over): {y.mean():.3f}")
print(f"🎯 Baseline (sempre Under): {1-y.mean():.3f}")

In [ ]:
# Comparação visual
fig, ax = plt.subplots(figsize=(8, 4))
names = list(results.keys())
means = [results[n]["mean"] for n in names]
stds = [results[n]["std"] for n in names]
colors = ["#1565C0", "#2e7d32", "#E65100"][:len(names)]

bars = ax.bar(names, means, yerr=stds, capsize=5, color=colors, edgecolor="white", alpha=0.85)
ax.axhline(max(y.mean(), 1-y.mean()), color="red", linestyle="--", alpha=0.5, label=f"Baseline: {max(y.mean(), 1-y.mean()):.3f}")

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{mean:.1%}", ha="center", fontweight="bold", fontsize=12)

ax.set_ylim(0.3, 1.0)
ax.set_ylabel("Accuracy")
ax.set_title("Comparação de Modelos — Over/Under")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Análise detalhada do melhor modelo

In [ ]:
# Treinar o melhor modelo no dataset completo e analisar
best_name = max(results, key=lambda n: results[n]["mean"])
print(f"Melhor modelo: {best_name} ({results[best_name]['mean']:.3f})")

best_model = models[best_name]
best_model.fit(X, y)

# Feature importance
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
elif hasattr(best_model, "named_steps") and hasattr(best_model.named_steps.get("clf", None), "coef_"):
    importances = np.abs(best_model.named_steps["clf"].coef_[0])
else:
    importances = None

if importances is not None:
    feat_imp = pd.Series(importances, index=model_features).sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(8, 10))
    feat_imp.tail(20).plot(kind="barh", ax=ax, color="#1565C0", edgecolor="white")
    ax.set_title(f"Top 20 Features — {best_name}")
    ax.set_xlabel("Importância")
    plt.tight_layout()
    plt.show()

In [ ]:
# Confusion matrix com cross-validation predictions
from sklearn.model_selection import cross_val_predict

y_pred_cv = cross_val_predict(models[best_name], X, y, cv=cv)

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(
    y, y_pred_cv,
    display_labels=["Under", "Over"],
    cmap="Blues", ax=ax
)
ax.set_title(f"Matriz de Confusão — {best_name}")
plt.tight_layout()
plt.show()

print(f"\n{classification_report(y, y_pred_cv, target_names=['Under', 'Over'])}")

## 8. Abordagem alternativa: Regressão (prever total)

Em vez de classificar Over/Under diretamente, prever o **total de pontos** como valor contínuo.  
Depois comparar com a linha para decidir Over/Under — mais flexível, funciona com qualquer linha.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_predict

y_total = df_ml["total_pts"]

reg_models = {
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("reg", Ridge(alpha=1.0))
    ]),
    "Random Forest Reg": RandomForestRegressor(
        n_estimators=100, max_depth=5, random_state=42, n_jobs=-1
    ),
}

cv_reg = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in reg_models.items():
    y_pred = cross_val_predict(model, X, y_total, cv=cv_reg)
    mae = mean_absolute_error(y_total, y_pred)
    rmse = np.sqrt(mean_squared_error(y_total, y_pred))
    
    # Converter para Over/Under
    y_ou_pred = (y_pred > LINE).astype(int)
    acc = accuracy_score(y, y_ou_pred)
    
    print(f"{name:25s}  MAE: {mae:.1f} pts | RMSE: {rmse:.1f} pts | O/U Acc: {acc:.3f}")

In [ ]:
# Visualizar previsão vs real (Ridge)
ridge = reg_models["Ridge Regression"]
y_pred_ridge = cross_val_predict(ridge, X, y_total, cv=cv_reg)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: predito vs real
axes[0].scatter(y_total, y_pred_ridge, alpha=0.5, s=30, color="#1565C0")
lims = [min(y_total.min(), y_pred_ridge.min()) - 5, max(y_total.max(), y_pred_ridge.max()) + 5]
axes[0].plot(lims, lims, "--", color="red", alpha=0.5)
axes[0].set_xlabel("Total real")
axes[0].set_ylabel("Total predito")
axes[0].set_title("Ridge — Predito vs Real")

# Distribuição dos erros
errors = y_total.values - y_pred_ridge
axes[1].hist(errors, bins=20, color="#2e7d32", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Erro (real - predito)")
axes[1].set_ylabel("Frequência")
axes[1].set_title(f"Distribuição dos erros (MAE: {mean_absolute_error(y_total, y_pred_ridge):.1f})")

plt.tight_layout()
plt.show()

## 9. Resumo e próximos passos

### O que aprendemos
- Quais features são mais preditivas do total de pontos
- Qual modelo teve melhor performance inicial
- Limitação principal: **poucas amostras** (~225 jogos de uma temporada)

### Para melhorar a acurácia
1. **Mais dados**: buscar temporadas anteriores (2022-23, 2023-24, 2024-25) — multiplicar por 4x o dataset
2. **Rolling averages**: usar média dos últimos 5-10 jogos em vez da média da temporada inteira
3. **Rest days**: calcular dias de descanso entre jogos
4. **Back-to-back**: flag para jogos em dias consecutivos
5. **Player impact**: incluir status de jogadores-chave (lesões)
6. **Vegas lines reais**: usar linhas reais em vez da mediana

### Integração com o dashboard
1. Salvar modelo treinado: `joblib.dump(model, 'data/ou_model.pkl')`
2. No `app.py`, carregar e fazer inferência em tempo real
3. Exibir: probabilidade Over/Under + confiança do modelo

In [ ]:
# Salvar o melhor modelo (descomente quando estiver satisfeito)

# import joblib
# best_model.fit(X, y)
# joblib.dump(best_model, "data/ou_model.pkl")
# joblib.dump(model_features, "data/ou_features.pkl")
# print("✅ Modelo salvo em data/ou_model.pkl")